**Test functions in deblur flow**
- Folder name can be set according to your need
- TEST_PAT_SIZE can be assigned to either **"small"** or **"large"**
  - **"small"** stands for small test pattern size
  - **"large"** stands for large test pattern size
  - During implementation, it is recommended to set TEST_PAT_SIZE **"small"** for quick debugging. However, you have to pass the unit test with both TEST_PAT_SIZE **"small"** and **"large"** to get the full score in each part.
  

In [ ]:
from pathlib import Path
import os


def find_hw02_root():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / 'code' / 'deblur_functions.ipynb').exists() and (path / 'data').exists():
            return Path(os.path.relpath(path, cwd))

        hw02 = path / 'hw02'
        if (hw02 / 'code' / 'deblur_functions.ipynb').exists() and (hw02 / 'data').exists():
            return Path(os.path.relpath(hw02, cwd))

    raise FileNotFoundError('Run this notebook from the repository root, hw02, or hw02/code.')


FOLDER_NAME = find_hw02_root()

TEST_PAT_SIZE = 'small'
# TEST_PAT_SIZE = 'large'


Do not modify the remaining code.

In [ ]:
from PIL import Image

import time
import numpy as np
import imageio
import matplotlib.pyplot as plt
import scipy.signal
from tqdm import tqdm
import sys

INF = float("inf")
DBL_MIN = sys.float_info.min


In [ ]:
%run "{FOLDER_NAME}/code/TV_functions.ipynb"
%run "{FOLDER_NAME}/code/deblur_functions.ipynb"

In [ ]:
def PSNR_UCHAR3(input_1, input_2, peak=255):
    [row,col,channel] = input_1.shape

    if input_1.shape != input_2.shape:
        print ("Warning!! Two image have different shape!!")
        return 0

    input_1 = input_1.astype('float')
    input_2 = input_2.astype('float')

    mse = ((input_1 - input_2)**2).sum() / float(row * col * channel)

    # print('mse: ', mse)
    if mse == 0.0:
        psnr = INF  # avoid divide zero case
    else:
        psnr = 10 * np.log10((255.0 ** 2)/mse)

    return psnr


In [ ]:
def Evaluate_PSNR(psnr, duration, target_psnr=60.0):
    print(f'    -> processing time = {duration:.2f} sec, PSNR = {psnr} dB')

    if(psnr<target_psnr):
        print('    -> status: \033[1;31;40m fail \033[0;0m ... QQ\n')
    else:
        print('    -> status: \033[1;32;40m pass \033[0;0m !!\n')


In [ ]:
def Evaluate_error(error, duration):
    print(f'    -> processing time = {duration:.2f} sec, error = {error:.4f} %')

    if(error>0.05):
        print('    -> status: \033[1;31;40m fail \033[0;0m ... QQ\n')
    else:
        print('    -> status: \033[1;32;40m pass \033[0;0m !!\n')


In [ ]:
def test_Wiener_deconv():
    print ("//--------------------------------------------------------")
    print (f"start Wiener deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/Wiener_m_SNRF100.0.png'))

    # setting
    SNR_F = 100.0

    # work
    t_start = time.time()
    Wiener_result = Wiener(img_in, k_in, SNR_F)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/Wiener_m_SNRF100.0.png' , Wiener_result)

    # evaluate
    psnr = PSNR_UCHAR3(Wiener_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration)

In [ ]:
def test_RL_a():
    print ("//--------------------------------------------------------")
    print (f"start RL-(a) deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_small.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_small.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_small/RL_s_iter30.png'))

    # setting
    max_iter_RL = 30

    # work
    t_start = time.time()
    RL_result = RL(img_in, k_in, max_iter_RL)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_small/RL_s_iter30.png' , RL_result)

    # evaluate
    psnr = PSNR_UCHAR3(RL_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration)

In [ ]:
def test_RL_b():
    print ("//--------------------------------------------------------")
    print (f"start RL-(b) deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/RL_m_iter45.png'))

    # setting
    max_iter_RL = 45

    # work
    t_start = time.time()
    RL_result = RL(img_in, k_in, max_iter_RL)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/RL_m_iter45.png' , RL_result)

    # evaluate
    psnr = PSNR_UCHAR3(RL_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration)

In [ ]:
def test_RL_energy():
    print ("//--------------------------------------------------------")
    print (f"start RL-energy function, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    blur_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_small.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_small.png'))
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_small/RL_s_iter30.png'))

    energy_dict = np.load(f'{FOLDER_NAME}/golden/energy_dict.npy',allow_pickle='TRUE').item()
    golden = energy_dict[f'{TEST_PAT_SIZE}_RL_a']


    # work
    t_start = time.time()
    energy = check_RL_energy(img_in, k_in, blur_in)
    t_end = time.time()

    # evaluate
    print(f'RL energy: {energy}, golden energy: {golden}')
    duration = t_end - t_start
    Evaluate_error( abs((energy-golden)/golden)*100, duration)

In [ ]:
def test_BRL_a():
    print ("//--------------------------------------------------------")
    print (f"start BRL-(a) deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_small.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_small.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_small/BRL_s_iter15_rk6_si50.00_lam0.030.png'))

    # setting
    max_iter_RL = 15
    rk = 6
    sigma_r = 50.0/255/255
    lamb_da = 0.03/255

    # work
    t_start = time.time()
    BRL_result = BRL(img_in, k_in, max_iter_RL, lamb_da, sigma_r, rk)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_small/BRL_s_iter15_rk6_si50.00_lam0.030.png' , BRL_result)

    # evaluate
    psnr = PSNR_UCHAR3(BRL_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration, target_psnr=55.0)

In [ ]:
def test_BRL_b():
    print ("//--------------------------------------------------------")
    print (f"start BRL-(b) deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_small.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_small.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_small/BRL_s_iter15_rk6_si50.00_lam0.060.png'))

    # setting
    max_iter_RL = 15
    rk = 6
    sigma_r = 50.0/255/255
    lamb_da = 0.06/255

    # work
    t_start = time.time()
    BRL_result = BRL(img_in, k_in, max_iter_RL, lamb_da, sigma_r, rk)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_small/BRL_s_iter15_rk6_si50.00_lam0.060.png' , BRL_result)

    # evaluate
    psnr = PSNR_UCHAR3(BRL_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration, target_psnr=55.0)

In [ ]:
def test_BRL_c():
    print ("//--------------------------------------------------------")
    print (f"start BRL-(c) deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/BRL_m_iter40_rk12_si25.00_lam0.001.png'))

    # setting
    max_iter_RL = 40
    rk = 12
    sigma_r = 25.0/255/255
    lamb_da = 0.001/255

    # work
    t_start = time.time()
    BRL_result = BRL(img_in, k_in, max_iter_RL, lamb_da, sigma_r, rk)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/BRL_m_iter40_rk12_si25.00_lam0.001.png' , BRL_result)

    # evaluate
    psnr = PSNR_UCHAR3(BRL_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration, target_psnr=55.0)

In [ ]:
def test_BRL_d():
    print ("//--------------------------------------------------------")
    print (f"start BRL-(d) deconvolution, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/BRL_m_iter40_rk12_si25.00_lam0.006.png'))

    # setting
    max_iter_RL = 40
    rk = 12
    sigma_r = 25.0/255/255
    lamb_da = 0.006/255

    # work
    t_start = time.time()
    BRL_result = BRL(img_in, k_in, max_iter_RL, lamb_da, sigma_r, rk)
    t_end = time.time()
    psnr = PSNR_UCHAR3(BRL_result, img_golden)

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/BRL_m_iter40_rk12_si25.00_lam0.006.png' , BRL_result)

    # evaluate
    psnr = PSNR_UCHAR3(BRL_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration, target_psnr=55.0)

In [ ]:
def test_BRL_energy():
    print ("//--------------------------------------------------------")
    print (f"start BRL-energy function, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    blur_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_small.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_small.png'))
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_small/BRL_s_iter15_rk6_si50.00_lam0.030.png'))

    energy_dict = np.load(f'{FOLDER_NAME}/golden/energy_dict.npy',allow_pickle='TRUE').item()
    golden = energy_dict[f'{TEST_PAT_SIZE}_BRL_a']


    # setting
    rk = 6
    sigma_r = 50.0/255/255
    lamb_da = 0.03/255

    # work
    t_start = time.time()
    energy = BRL_energy(img_in, k_in, blur_in, lamb_da, sigma_r, rk)
    t_end = time.time()

    # evaluate
    print(f'BRL energy: {energy}, golden energy: {golden}')
    duration = t_end - t_start
    Evaluate_error( abs((energy-golden)/golden)*100, duration)


In [ ]:
def test_TVL1():
    print ("//--------------------------------------------------------")
    print (f"start TVL1, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/TVL1_m_iter1000_lam0.010.png'))

    # setting
    max_iter = 1000
    lamb_da = 0.01

    # work
    t_start = time.time()
    TVL1_result = TVL1(img_in, k_in, max_iter, lamb_da)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/TVL1_m_iter1000_lam0.010.png' , TVL1_result)

    # evaluate
    psnr = PSNR_UCHAR3(TVL1_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration)

In [ ]:
def test_TVL2():
    print ("//--------------------------------------------------------")
    print (f"start TVL2, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/TVL2_m_iter1000_lam0.010.png'))

    # setting
    max_iter = 1000
    lamb_da = 0.01

    # work
    t_start = time.time()
    TVL2_result = TVL2(img_in, k_in, max_iter, lamb_da)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/TVL2_m_iter1000_lam0.010.png' , TVL2_result)

    # evaluate
    psnr = PSNR_UCHAR3(TVL2_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration)

In [ ]:
def test_TVpoisson():
    print ("//--------------------------------------------------------")
    print (f"start TVpoisson, TEST_PAT_SIZE: {TEST_PAT_SIZE}\n")

    # I/O
    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))
    img_golden = np.asarray(Image.open(f'{FOLDER_NAME}/golden/golden_{TEST_PAT_SIZE}/curiosity_medium/TVpoisson_m_iter1000_lam0.010.png'))

    # setting
    max_iter = 1000
    lamb_da = 0.01

    # work
    t_start = time.time()
    TVpoisson_result = TVpoisson(img_in, k_in, max_iter, lamb_da)
    t_end = time.time()

    # store image
    imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/TVpoisson_m_iter1000_lam0.010.png' , TVpoisson_result)

    # evaluate
    psnr = PSNR_UCHAR3(TVpoisson_result, img_golden)
    duration = t_end - t_start
    Evaluate_PSNR(psnr, duration)

In [ ]:
## (1) Wiener part
test_Wiener_deconv()


## (2) RL part
test_RL_a()
test_RL_b()
test_RL_energy()


## (3) BRL part
test_BRL_a()
test_BRL_b()
test_BRL_c()
test_BRL_d()
test_BRL_energy()


## (4) Total variation part
test_TVL1() # already done for reference, but you have to first finish img_conversion and kernel_conversion in deblur_functions.ipynb
test_TVL2()
test_TVpoisson()

In [ ]:
# 2.1.a
def get_dft_curve(img):

  scanline = np.mean(img, axis = 2)
  fft_scanline = np.fft.rfft(scanline, axis = 1)
  log_mag = np.log10(np.mean(np.abs(fft_scanline), axis = 0) + DBL_MIN)

  return log_mag

def plot_dft_result():

  img_true = np.asarray(Image.open(f'{FOLDER_NAME}/data/curiosity.png'))
  img_blurred = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_large/curiosity_medium.png'))

  img_Wiener = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/Wiener_m_SNRF100.0.png'))
  img_RL = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/RL_m_iter45.png'))
  img_BRL = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/BRL_m_iter40_rk12_si25.00_lam0.001.png'))
  img_TVL1 = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVL1_m_iter1000_lam0.010.png'))
  img_TVL2 = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVL2_m_iter1000_lam0.010.png'))
  img_TVpoisson = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVpoisson_m_iter1000_lam0.010.png'))

  log_mag_true = get_dft_curve(img_true)
  log_mag_blurred = get_dft_curve(img_blurred)

  log_mag_Wiener = get_dft_curve(img_Wiener)
  log_mag_RL = get_dft_curve(img_RL)
  log_mag_BRL = get_dft_curve(img_BRL)
  log_mag_TVL1 = get_dft_curve(img_TVL1)
  log_mag_TVL2 = get_dft_curve(img_TVL2)
  log_mag_TVLpoisson = get_dft_curve(img_TVpoisson)

  PSNR_blurred = PSNR_UCHAR3(img_true, img_blurred)
  PSNR_Wiener = PSNR_UCHAR3(img_true, img_Wiener)
  PSNR_RL = PSNR_UCHAR3(img_true, img_RL)
  PSNR_BRL = PSNR_UCHAR3(img_true, img_BRL)
  PSNR_TVL1 = PSNR_UCHAR3(img_true, img_TVL1)
  PSNR_TVL2 = PSNR_UCHAR3(img_true, img_TVL2)
  PSNR_TVpoisson = PSNR_UCHAR3(img_true, img_TVpoisson)


  plt.subplots(figsize = (8, 6))
  freqs = np.linspace(0, np.pi, len(log_mag_true))

  plt.plot(freqs, log_mag_true, label = "true image", linewidth = 1)
  plt.plot(freqs, log_mag_blurred, label = f"blurred image (PSNR: {PSNR_blurred:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_Wiener, label = f"Wiener (PSNR: {PSNR_Wiener:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_RL, label = f"RL (PSNR: {PSNR_RL:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_BRL, label = f"BRL (PSNR: {PSNR_BRL:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_TVL1, label = f"TVL1 (PSNR: {PSNR_TVL1:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_TVL2, label = f"TVL2 (PSNR: {PSNR_TVL2:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_TVLpoisson, label = f"TVpoisson (PSNR: {PSNR_TVpoisson:.4f})", linewidth = 1)

  plt.xlabel('Frequency')
  plt.ylabel('Log Magnitude')

  plt.xticks(
      [0, np.pi/5, 2*np.pi/5, 3*np.pi/5, 4*np.pi/5, np.pi],
      ['0', r'$\pi/5$', r'$2\pi/5$', r'$3\pi/5$', r'$4\pi/5$', r'$\pi$']
  )

  plt.xlim(0, np.pi)
  plt.legend(loc = 'upper right')
  plt.show()

plot_dft_result()


In [42]:
# 2.1.b
def plot_RL_energy():

    img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_large/curiosity_medium.png'))
    k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))

    max_iter_RL = 45
    RL_energies = []

    for i in tqdm(range(1, max_iter_RL + 1)):

      RL_result = RL(img_in, k_in, i)
      RL_energies.append(check_RL_energy(img_in, k_in, RL_result))

    plt.subplots(figsize = (8, 6))
    plt.plot(RL_energies)
    plt.xlabel('Iterations')
    plt.ylabel('RL energy')
    plt.show()

plot_RL_energy()

  9%|▉         | 4/45 [00:51<08:46, 12.84s/it]


KeyboardInterrupt: 

In [ ]:
# 2.1.d - kernel_1
def deblur_my_image_1(file_name):

  img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}.JPG'))
  img_in = np.clip((img_in / 255.0) ** 2.2, 0.0, 1.0)

  k_in = np.mean(np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}_kernel.JPG')), axis = 2)
  k_in = np.clip((k_in / 255.0) ** 2.2, 0.0, 1.0)

  max_iter_RL = 20
  rk = 12
  sigma_r = 25/255/255
  lamb_da = 0.05/255

  BRL_result = BRL(img_in, k_in, max_iter_RL, lamb_da, sigma_r, rk)

  BRL_result = (np.round(np.clip((BRL_result / 255.0) ** (1 / 2.2), 0.0, 1.0) * 255)).astype(np.uint8)

  imageio.imwrite(f'{FOLDER_NAME}/MyDeblur_result/BRL1_{file_name}_m_iter{max_iter_RL}_rk{rk}_si{sigma_r * 255 * 255: .2f}_lam{lamb_da * 255: .4f}.png', BRL_result)

deblur_my_image_1("7M506652") # Strong blur
deblur_my_image_1("7M506653") # Weak blur

In [ ]:
# 2.1.d - kernel_2
def deblur_my_image_2(file_name):

  img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}.JPG'))
  img_in = np.clip((img_in / 255.0) ** 2.2, 0.0, 1.0)

  k_in = np.mean(np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}_kernel.JPG')), axis = 2)
  k_in = np.where(k_in < 0.01, 0.0, k_in)
  k_in = np.clip((k_in / 255.0) ** 2.2, 0.0, 1.0)

  max_iter_RL = 20
  rk = 12
  sigma_r = 25/255/255
  lamb_da = 0.05/255

  BRL_result = BRL(img_in, k_in, max_iter_RL, lamb_da, sigma_r, rk)

  BRL_result = (np.round(np.clip((BRL_result / 255.0) ** (1 / 2.2), 0.0, 1.0) * 255)).astype(np.uint8)

  imageio.imwrite(f'{FOLDER_NAME}/MyDeblur_result/BRL2_{file_name}_m_iter{max_iter_RL}_rk{rk}_si{sigma_r * 255 * 255: .2f}_lam{lamb_da * 255: .4f}.png', BRL_result)

deblur_my_image_2("7M506652") # Strong blur
deblur_my_image_2("7M506653") # Weak blur

In [ ]:
# 2.1.d - kernel_3
def deblur_my_image_3(file_name):

  img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}.JPG'))
  k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}_kernel.JPG'))

  img_in = np.clip((img_in / 255.0) ** 2.2, 0.0, 1.0)
  k_in = np.clip((k_in / 255.0) ** 2.2, 0.0, 1.0)
  k = k_in / np.sum(k_in, axis = (0, 1))

  max_iter_RL = 20
  rk = 12
  sigma_r = 25/255/255
  lamb_da = 0.05/255

  b = img_conversion(img_in, True)
  h, w, ch = b.shape
  k_adj = k[::-1, ::-1, :]

  I_x = b.copy()

  r_omega = rk // 2
  sigma_s = (r_omega / 3.0) ** 2

  pad_h = k.shape[0] // 2
  pad_w = k.shape[1] // 2

  for i in tqdm(range(max_iter_RL)):

    I_pad = np.pad(I_x, pad_width = ((r_omega, r_omega), (r_omega, r_omega), (0, 0)), mode = 'symmetric')
    grad_EB = np.zeros_like(I_x)

    for dy in range(-r_omega, r_omega + 1):
      for dx in range(-r_omega, r_omega + 1):

        f = np.exp(-(dy ** 2 + dx ** 2) / (2 * sigma_s))

        I_y = I_pad[r_omega + dy : r_omega + dy + h, r_omega + dx : r_omega + dx + w, :]
        g = np.exp(-((I_x - I_y) ** 2) / (2 * sigma_r))

        grad_EB += f * g * (I_x - I_y)

    grad_EB *= 2 / sigma_r

    # fftconvolve

    I_x_pad = np.pad(I_x, pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric')
    pred_B = scipy.signal.fftconvolve(I_x_pad, k, mode = 'valid', axes = (0, 1)) # I ⊗ K
    pred_B = np.maximum(pred_B, DBL_MIN)
    ratio = np.pad(b / (pred_B + DBL_MIN), pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric') # B / (I ⊗ K)
    error = scipy.signal.fftconvolve(ratio, k_adj, mode = 'valid', axes = (0, 1)) # K* ⊗ (B / (I ⊗ K))

    I_x = (I_x / (1 + lamb_da * grad_EB)) * error

  BRL_result = img_conversion(I_x, False)

  BRL_result = (np.round(np.clip((BRL_result / 255.0) ** (1 / 2.2), 0.0, 1.0) * 255)).astype(np.uint8)

  imageio.imwrite(f'{FOLDER_NAME}/MyDeblur_result/BRL3_{file_name}_m_iter{max_iter_RL}_rk{rk}_si{sigma_r * 255 * 255: .2f}_lam{lamb_da * 255: .4f}.png', BRL_result)

deblur_my_image_3("7M506652") # Strong blur
deblur_my_image_3("7M506653") # Weak blur

In [ ]:
# 2.1.d - kernel_4
def deblur_my_image_4(file_name):

  img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}.JPG'))
  img_in = np.clip((img_in / 255.0) ** 2.2, 0.0, 1.0)

  k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/{file_name}_kernel.JPG'))
  k_in = np.where(k_in < 0.01, 0.0, k_in)
  k_in = np.clip((k_in / 255.0) ** 2.2, 0.0, 1.0)
  k = k_in / np.sum(k_in, axis = (0, 1))

  max_iter_RL = 20
  rk = 12
  sigma_r = 25/255/255
  lamb_da = 0.05/255

  b = img_conversion(img_in, True)
  h, w, ch = b.shape
  k_adj = k[::-1, ::-1, :]

  I_x = b.copy()

  r_omega = rk // 2
  sigma_s = (r_omega / 3.0) ** 2

  pad_h = k.shape[0] // 2
  pad_w = k.shape[1] // 2

  for i in tqdm(range(max_iter_RL)):

    I_pad = np.pad(I_x, pad_width = ((r_omega, r_omega), (r_omega, r_omega), (0, 0)), mode = 'symmetric')
    grad_EB = np.zeros_like(I_x)

    for dy in range(-r_omega, r_omega + 1):
      for dx in range(-r_omega, r_omega + 1):

        f = np.exp(-(dy ** 2 + dx ** 2) / (2 * sigma_s))

        I_y = I_pad[r_omega + dy : r_omega + dy + h, r_omega + dx : r_omega + dx + w, :]
        g = np.exp(-((I_x - I_y) ** 2) / (2 * sigma_r))

        grad_EB += f * g * (I_x - I_y)

    grad_EB *= 2 / sigma_r

    # fftconvolve

    I_x_pad = np.pad(I_x, pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric')
    pred_B = scipy.signal.fftconvolve(I_x_pad, k, mode = 'valid', axes = (0, 1)) # I ⊗ K
    pred_B = np.maximum(pred_B, DBL_MIN)
    ratio = np.pad(b / (pred_B + DBL_MIN), pad_width = ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode = 'symmetric') # B / (I ⊗ K)
    error = scipy.signal.fftconvolve(ratio, k_adj, mode = 'valid', axes = (0, 1)) # K* ⊗ (B / (I ⊗ K))

    I_x = (I_x / (1 + lamb_da * grad_EB)) * error

  BRL_result = img_conversion(I_x, False)

  BRL_result = (np.round(np.clip((BRL_result / 255.0) ** (1 / 2.2), 0.0, 1.0) * 255)).astype(np.uint8)

  imageio.imwrite(f'{FOLDER_NAME}/MyDeblur_result/BRL4_{file_name}_m_iter{max_iter_RL}_rk{rk}_si{sigma_r * 255 * 255: .2f}_lam{lamb_da * 255: .4f}.png', BRL_result)

deblur_my_image_4("7M506652") # Strong blur
deblur_my_image_4("7M506653") # Weak blur

In [ ]:
# 2,2,a

def Wiener_without_rollng(img_in, k_in, SNR_F):

  b = img_conversion(img_in, True)
  img_h, img_w, ch = b.shape

  k_h, k_w = k_in.shape
  k_pad = np.zeros((img_h, img_w), dtype = 'uint8')
  k_pad[0:k_h, 0:k_w] = k_in
  k = kernel_conversion(k_pad)
  K = np.fft.rfft2(k)

  Wiener_result = np.empty(b.shape, dtype = 'uint8')

  for c in range(ch):

    B = np.fft.rfft2(b[:, :, c])

    I = B * (np.conj(K) / (np.abs(K) ** 2 + (1 / SNR_F)))
    I_hat = np.fft.irfft2(I)

    Wiener_result[:, :, c] = img_conversion(I_hat, False)

  return Wiener_result

def test_Wiener_without_rollng():

  img_true = np.asarray(Image.open(f'{FOLDER_NAME}/data/curiosity.png'))
  img_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_{TEST_PAT_SIZE}/curiosity_medium.png'))
  k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))

  SNR_F = 100
  Wiener_without_rollng_result = Wiener_without_rollng(img_in, k_in, SNR_F)
  Wiener_result = Wiener(img_in, k_in, SNR_F)

  imageio.imwrite(f'{FOLDER_NAME}/result/result_{TEST_PAT_SIZE}/curiosity_medium/Wiener_without_rolling_m_SNRF100.0.png' , Wiener_without_rollng_result)

  PSNR_blurred = PSNR_UCHAR3(img_true, img_in)
  PSNR_Wiener = PSNR_UCHAR3(img_true, Wiener_result)
  PSNR_Wiener_without_rolling = PSNR_UCHAR3(img_true, Wiener_without_rollng_result)

  log_mag_true = get_dft_curve(img_true)
  log_mag_blurred = get_dft_curve(img_in)

  log_mag_Wiener = get_dft_curve(Wiener_result)
  log_mag_Wiener_without_rolling = get_dft_curve(Wiener_without_rollng_result)

  plt.subplots(figsize = (8, 6))
  freqs = np.linspace(0, np.pi, len(log_mag_true))

  plt.plot(freqs, log_mag_true, label = "true image", linewidth = 1)
  plt.plot(freqs, log_mag_blurred, label = f"blurred image (PSNR: {PSNR_blurred:.4f})", linewidth = 1)
  plt.plot(freqs, log_mag_Wiener, label = f"Wiener (PSNR: {PSNR_Wiener:.4f})", linewidth = 2.5)
  plt.plot(freqs, log_mag_Wiener_without_rolling, label = f"Wiener without rolling (PSNR: {PSNR_Wiener_without_rolling:.4f})", linewidth = 1)

  plt.xticks(
      [0, np.pi/5, 2*np.pi/5, 3*np.pi/5, 4*np.pi/5, np.pi],
      ['0', r'$\pi/5$', r'$2\pi/5$', r'$3\pi/5$', r'$4\pi/5$', r'$\pi$']
  )

  plt.xlim(0, np.pi)
  plt.legend(loc = 'upper right')
  plt.show()

test_Wiener_without_rollng()

In [ ]:
# 2.2.c
def sigma_params_test():

  img_true = np.asarray(Image.open(f'{FOLDER_NAME}/data/curiosity.png'))
  img_blur = np.asarray(Image.open(f'{FOLDER_NAME}/data/blurred_image_large/curiosity_medium.png'))
  k_in = np.asarray(Image.open(f'{FOLDER_NAME}/data/kernel/kernel_medium.png'))

  sigmas = [1e-4, 1e-3, 1e-2]
  lambdas = [0.001, 0.01, 0.1]
  iterations = [500, 1000, 1500]

  img_list = []

  img_blur = img_conversion(img_blur, True)

  for sigma in sigmas:

    noise = np.random.normal(0, sigma, img_blur.shape)
    img_noise = np.clip(img_blur + noise, 0.0, 1.0)
    img_list.append(img_noise)


  psnr_sigma_lambda = [[0 for _ in range(3)] for _ in range(3)]

  for i, sigma in enumerate(sigmas):
    for j, lamb_da in enumerate(lambdas):

      try:
        img_deblurred = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVL1_sigma{sigma}_lambda{lamb_da}.png'))

      except:
        img_deblurred = TVL1(img_list[i], k_in, 1000, lamb_da)
        imageio.imwrite(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVL1_sigma{sigma}_lambda{lamb_da}.png', img_deblurred)

      psnr = PSNR_UCHAR3(img_true, img_deblurred)
      psnr_sigma_lambda[i][j] = psnr

      print(f"-- Sigma = {sigma}, Lambda = {lamb_da}, PSNR = {psnr:.4f}]")

  psnr_sigma_iteration = [[0 for _ in range(3)] for _ in range(3)]

  for i, sigma in enumerate(sigmas):
    for j, iter in enumerate(iterations):

      try:
        img_deblurred = np.asarray(Image.open(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVL1_sigma{sigma}_iter{iter}.png'))
      except:
        img_deblurred = TVL1(img_list[i], k_in, iter, 0.01)
        imageio.imwrite(f'{FOLDER_NAME}/result/result_large/curiosity_medium/TVL1_sigma{sigma}_iter{iter}.png', img_deblurred)

      psnr = PSNR_UCHAR3(img_true, img_deblurred)
      psnr_sigma_iteration[i][j] = psnr

      print(f"-- Sigma = {sigma}, Iteration = {iter}, PSNR = {psnr:.4f}]")

sigma_params_test()